
# KAMP Injection Molding — v6 Novelty Extension
## Run-Aware + Identifiability-Aware Hierarchical Inspection

This notebook combines the strongest ideas from:
- the existing v5 identifiability / ambiguity / external-context analysis, and
- the team's run-aware inspection report methodology.

### New v6 contributions
1. Full KAMP raw labeled data → **shot-level aggregation**
2. **30-minute run segmentation**
3. **run-aware feature engineering**
4. **sign-constrained logistic risk model**
5. **LOGO (leave-one-run-out) + sequential validation**
6. **60-minute rule + risk hybrid inspection policy**
7. **learned LH/RH side prioritization**
8. **split-conformal selective side inspection**
9. **hierarchical inspection policy**
   - Stage 1: risky shot?
   - Stage 2: which side?
   - Stage 3: uncertain → inspect both
10. **Value-of-Information (VOI)** for additional context
11. Optional RWTH cavity-pressure external-data profiler

The goal is not to replace v5, but to test whether the combined framework gives a
clearer originality story before the final clean run.


In [1]:

# ============================================================
# 0. Setup
# ============================================================
!pip -q install openpyxl pyarrow

import os, re, math, json, hashlib, shutil, zipfile, subprocess, urllib.request, warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

WORKDIR = Path("/content/kamp_v6_novelty")
OUT_DIR = WORKDIR / "outputs"
PLOT_DIR = OUT_DIR / "plots"
EXT_DIR = WORKDIR / "external"

for p in [WORKDIR, OUT_DIR, PLOT_DIR, EXT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LEGACY_KAMP_REPO = "https://github.com/johnwslee/injection_molding_analysis.git"
RWTH_URL = "https://publications.rwth-aachen.de/record/1016199/files/ExperimentalData.zip"

# Run switches
RUN_RUN_AWARE = True
RUN_SIDE_MODEL = True
RUN_CONFORMAL = True
RUN_HIERARCHICAL_POLICY = True

# Optional external profiler: turn on only after the core v6 sections work.
AUTO_DOWNLOAD_RWTH = False

# Inspection budgets
BUDGETS = [0.10, 0.20, 0.30, 0.40]

print("WORKDIR:", WORKDIR)


WORKDIR: /content/kamp_v6_novelty



## 1. Load full/legacy KAMP raw labeled data

The public legacy KAMP repository contains the raw labeled table with timestamp,
part name (including LH/RH), shot information, and process variables.

The loader intentionally prefers `data/raw/labeled_data.csv`.


In [2]:

# ============================================================
# 1. Clone / load legacy KAMP
# ============================================================
def canonical_name(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def read_csv_flexible(path):
    for enc in ["utf-8", "utf-8-sig", "cp949", "euc-kr", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    raise RuntimeError(f"Could not read {path}")

repo = EXT_DIR / "legacy_kamp"

if repo.exists():
    shutil.rmtree(repo)

subprocess.run(
    ["git", "clone", "--depth", "1", LEGACY_KAMP_REPO, str(repo)],
    check=True
)

preferred = repo / "data" / "raw" / "labeled_data.csv"

if preferred.exists():
    raw = read_csv_flexible(preferred)
    raw_path = preferred
else:
    candidates = []
    for p in repo.rglob("*.csv"):
        try:
            df0 = read_csv_flexible(p)
        except Exception:
            continue
        cc = {canonical_name(c) for c in df0.columns}
        if "passorfail" in cc:
            candidates.append((len(df0), p, df0))
    if not candidates:
        raise RuntimeError("No legacy labeled CSV containing PassOrFail found.")
    candidates.sort(reverse=True, key=lambda x: x[0])
    _, raw_path, raw = candidates[0]

print("Loaded:", raw_path)
print("Raw shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())


Loaded: /content/kamp_v6_novelty/external/legacy_kamp/data/raw/labeled_data.csv
Raw shape: (7996, 45)
Columns: ['_id', 'TimeStamp', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL', 'PART_NAME', 'EQUIP_CD', 'EQUIP_NAME', 'PassOrFail', 'Reason', 'Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Switch_Over_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Barrel_Temperature_7', 'Hopper_Temperature', 'Mold_Temperature_1', 'Mold_Temperature_2', 'Mold_Temperature_3', 'Mold_Temperature_4', 'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'Mold_Temperature_10', 'Mold_Tem

,_id,TimeStamp,PART_FACT_PLAN_DATE,PART_FACT_SERIAL,PART_NAME,EQUIP_CD,EQUIP_NAME,PassOrFail,Reason,Injection_Time,...,Mold_Temperature_3,Mold_Temperature_4,Mold_Temperature_5,Mold_Temperature_6,Mold_Temperature_7,Mold_Temperature_8,Mold_Temperature_9,Mold_Temperature_10,Mold_Temperature_11,Mold_Temperature_12
0,5f8928bb9c0189cc666ef19b,2020-10-16 04:57:47,2020-10-16 오전 12:00:00,24,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기,Y,NaN,9.59,...,24.799999,27.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5f8928de9c0189cc666ef20b,2020-10-16 04:58:48,2020-10-16 오전 12:00:00,24,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기,Y,NaN,9.60,...,24.799999,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5f8928df9c0189cc666ef213,2020-10-16 04:58:48,2020-10-16 오전 12:00:00,23,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기,Y,NaN,9.60,...,24.799999,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5f8928f39c0189cc666ef25e,2020-10-16 04:59:48,2020-10-16 오전 12:00:00,23,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기,Y,NaN,9.59,...,25.000000,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5f8928f59c0189cc666ef265,2020-10-16 04:59:48,2020-10-16 오전 12:00:00,24,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기,Y,NaN,9.59,...,25.000000,27.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



## 2. Reconstruct shot-level data and production runs

Report-inspired preprocessing:
- map `Y = Pass`, `N = Fail`
- remove exact duplicates
- repair the known ×10 `Average_Screw_RPM` recording regime when average > 2× max
- aggregate LH/RH part rows into one shot
- if either side fails, the shot is a fail
- segment a new run after a >30 minute gap within a product
- create elapsed-time and early-60-minute context

No future labels are used in run construction.


In [3]:

# ============================================================
# 2. Raw -> shot -> run
# ============================================================
df = raw.copy()

canon = {canonical_name(c): c for c in df.columns}

def col(name, required=True):
    c = canon.get(canonical_name(name))
    if required and c is None:
        raise KeyError(f"Required column not found: {name}")
    return c

TS = col("TimeStamp")
TARGET_RAW = col("PassOrFail")
PART_NAME = col("PART_NAME")
SHOT_NO = col("Shot_Number", required=False)
REASON = col("Reason", required=False)
IDCOL = col("_id", required=False)

PROCESS_FEATURES = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]

# Keep only available process features, but require substantial overlap.
PROCESS_FEATURES = [c for c in PROCESS_FEATURES if c in df.columns]
if len(PROCESS_FEATURES) < 20:
    raise RuntimeError(f"Only {len(PROCESS_FEATURES)} expected process features found.")

# 1) exact duplicate removal
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n1 = len(df)

# 2) timestamp
df["_timestamp"] = pd.to_datetime(df[TS], errors="coerce")
df = df[df["_timestamp"].notna()].copy()

# 3) target
txt = df[TARGET_RAW].astype(str).str.strip().str.upper()
df["_y"] = txt.map({
    "Y":0, "N":1,
    "PASS":0, "FAIL":1,
    "OK":0, "NG":1, "NOK":1,
})
if df["_y"].isna().all():
    raise RuntimeError("Could not parse PassOrFail.")

# 4) product + side from PART_NAME
ptext = df[PART_NAME].astype(str).str.upper()
df["_product"] = np.select(
    [ptext.str.contains("CN7", na=False),
     ptext.str.contains("RG3", na=False)],
    ["CN7","RG3"],
    default="OTHER"
)

df["_side"] = np.select(
    [ptext.str.contains(r"\bLH\b|LEFT", regex=True, na=False),
     ptext.str.contains(r"\bRH\b|RIGHT", regex=True, na=False)],
    ["LH","RH"],
    default="UNKNOWN"
)

df = df[df["_product"].isin(["CN7","RG3"])].copy()

# 5) numeric conversion
for c in PROCESS_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 6) known Average_Screw_RPM ×10 regime correction
if "Average_Screw_RPM" in df.columns and "Max_Screw_RPM" in df.columns:
    bad = (
        df["Average_Screw_RPM"].notna()
        & df["Max_Screw_RPM"].notna()
        & (df["Max_Screw_RPM"] > 0)
        & (df["Average_Screw_RPM"] > 2.0 * df["Max_Screw_RPM"])
    )
    df["_avg_screw_rpm_corrected"] = bad.astype(int)
    df.loc[bad, "Average_Screw_RPM"] = (
        df.loc[bad, "Average_Screw_RPM"] / 10.0
    )
else:
    df["_avg_screw_rpm_corrected"] = 0

# 7) shot key
if SHOT_NO is not None:
    shot_key_cols = ["_product", "_timestamp", SHOT_NO]
else:
    # robust fallback
    shot_key_cols = ["_product", "_timestamp"] + PROCESS_FEATURES

def fail_side(g):
    sides = set(g.loc[g["_y"]==1, "_side"].dropna())
    if "LH" in sides and "RH" in sides:
        return "BOTH"
    if "RH" in sides:
        return "RH"
    if "LH" in sides:
        return "LH"
    return "NONE"

agg_spec = {c:"first" for c in PROCESS_FEATURES}
agg_spec.update({
    "_y":"max",
    "_side":"first",
    PART_NAME:"first",
    "_avg_screw_rpm_corrected":"max",
})

if REASON is not None:
    agg_spec[REASON] = lambda x: "|".join(
        sorted(set(str(v) for v in x.dropna() if str(v).lower() not in ["nan","none",""]))
    )

shot = (
    df.groupby(shot_key_cols, dropna=False)
    .agg(agg_spec)
    .reset_index()
)

side_table = (
    df.groupby(shot_key_cols, dropna=False)
    .apply(fail_side)
    .rename("_fail_side")
    .reset_index()
)

shot = shot.merge(side_table, on=shot_key_cols, how="left")

# product/timestamp already in keys
shot = shot.sort_values(["_product","_timestamp"]).reset_index(drop=True)

# 8) run segmentation: >30 min gap within product
shot["_gap_min"] = (
    shot.groupby("_product")["_timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

shot["_new_run"] = (
    shot["_gap_min"].isna()
    | (shot["_gap_min"] > 30)
).astype(int)

shot["_run_seq"] = shot.groupby("_product")["_new_run"].cumsum() - 1
shot["run_id"] = (
    shot["_product"]
    + "_r"
    + shot["_run_seq"].astype(int).astype(str).str.zfill(2)
)

run_start = shot.groupby("run_id")["_timestamp"].transform("min")
shot["elapsed_min"] = (
    (shot["_timestamp"] - run_start)
    .dt.total_seconds()
    .div(60)
)
shot["log_elapsed_min"] = np.log1p(shot["elapsed_min"].clip(lower=0))
shot["early60"] = (shot["elapsed_min"] <= 60).astype(int)
shot["run_shot_index"] = shot.groupby("run_id").cumcount()
shot["product_RG3"] = (shot["_product"]=="RG3").astype(int)

audit = pd.DataFrame([{
    "raw_rows": n0,
    "after_exact_duplicate_removal": n1,
    "part_rows_used": len(df),
    "shot_rows": len(shot),
    "fail_shots": int(shot["_y"].sum()),
    "fail_rate": float(shot["_y"].mean()),
    "runs": int(shot["run_id"].nunique()),
    "cn7_runs": int(shot.loc[shot["_product"]=="CN7","run_id"].nunique()),
    "rg3_runs": int(shot.loc[shot["_product"]=="RG3","run_id"].nunique()),
    "avg_screw_rpm_corrected_part_rows": int(df["_avg_screw_rpm_corrected"].sum()),
}])

audit.to_csv(OUT_DIR/"V6_01_shot_run_audit.csv", index=False)
shot.to_csv(OUT_DIR/"V6_01_shot_table.csv", index=False)

display(audit)
display(
    shot.groupby(["_product","run_id"])
    .agg(
        start=("_timestamp","min"),
        end=("_timestamp","max"),
        shots=("_y","size"),
        fails=("_y","sum"),
    )
    .reset_index()
)
print("\nFail side:")
display(shot.loc[shot["_y"]==1, "_fail_side"].value_counts())


ValueError: cannot insert Mold_Temperature_4, already exists


## 3. Run-aware + manufacturing-knowledge features

We retain the 24 process variables and add:
- run context
- physically interpretable aggregate features
- causal-time-safe rolling features using prior shots only

This is a compact reproduction of the strongest feature ideas from the teammate report,
not an attempt to duplicate every one of its 90 variables.


In [ ]:

# ============================================================
# 3. Feature engineering
# ============================================================
def safe_div(a, b, eps=1e-6):
    return a / np.where(np.abs(b) < eps, np.nan, b)

feat = shot.copy()

# Aggregate process knowledge
barrel_cols = [c for c in PROCESS_FEATURES if c.startswith("Barrel_Temperature_")]
mold_cols = [c for c in ["Mold_Temperature_3","Mold_Temperature_4"] if c in feat.columns]

if barrel_cols:
    feat["barrel_mean"] = feat[barrel_cols].mean(axis=1)
    feat["barrel_std"] = feat[barrel_cols].std(axis=1)
    if "Barrel_Temperature_1" in feat and "Barrel_Temperature_6" in feat:
        feat["barrel_gradient_1_minus_6"] = (
            feat["Barrel_Temperature_1"] - feat["Barrel_Temperature_6"]
        )

if len(mold_cols) == 2:
    feat["mold_mean"] = feat[mold_cols].mean(axis=1)
    feat["mold_diff_3_minus_4"] = feat[mold_cols[0]] - feat[mold_cols[1]]

if "barrel_mean" in feat and "mold_mean" in feat:
    feat["melt_minus_mold_temp"] = feat["barrel_mean"] - feat["mold_mean"]

if {"Max_Injection_Pressure","Max_Injection_Speed"}.issubset(feat.columns):
    feat["viscosity_proxy_pressure_per_speed"] = safe_div(
        feat["Max_Injection_Pressure"].values,
        feat["Max_Injection_Speed"].values
    )

if {"Filling_Time","Injection_Time"}.issubset(feat.columns):
    feat["filling_to_injection_ratio"] = safe_div(
        feat["Filling_Time"].values,
        feat["Injection_Time"].values
    )

if {"Max_Switch_Over_Pressure","Max_Injection_Pressure"}.issubset(feat.columns):
    feat["switch_to_injection_pressure_ratio"] = safe_div(
        feat["Max_Switch_Over_Pressure"].values,
        feat["Max_Injection_Pressure"].values
    )

if {"Average_Back_Pressure","Max_Back_Pressure"}.issubset(feat.columns):
    feat["avg_to_max_back_pressure_ratio"] = safe_div(
        feat["Average_Back_Pressure"].values,
        feat["Max_Back_Pressure"].values
    )

if {"Plasticizing_Position","Plasticizing_Time"}.issubset(feat.columns):
    feat["plasticizing_position_per_sec"] = safe_div(
        feat["Plasticizing_Position"].values,
        feat["Plasticizing_Time"].values
    )

# Time-safe rolling features
ROLL_VARS = [
    c for c in [
        "Cushion_Position",
        "Filling_Time",
        "Max_Injection_Pressure",
        "Mold_Temperature_3",
        "Mold_Temperature_4",
    ]
    if c in feat.columns
]

for c in ROLL_VARS:
    prev = feat.groupby("run_id")[c].shift(1)
    feat[f"{c}__delta_prev"] = feat[c] - prev

    # previous five shots ONLY
    feat[f"{c}__prev5_range"] = (
        prev.groupby(feat["run_id"])
        .rolling(5, min_periods=2)
        .apply(lambda x: np.nanmax(x)-np.nanmin(x), raw=True)
        .reset_index(level=0, drop=True)
    )

# mold temperature slope using previous shot
if "mold_mean" in feat.columns:
    prev_mold = feat.groupby("run_id")["mold_mean"].shift(1)
    prev_time = feat.groupby("run_id")["elapsed_min"].shift(1)
    feat["mold_temp_slope_per_min"] = safe_div(
        (feat["mold_mean"] - prev_mold).values,
        (feat["elapsed_min"] - prev_time).values
    )

RUN_CONTEXT = [
    "elapsed_min","log_elapsed_min","early60",
    "run_shot_index","product_RG3"
]

DERIVED = [
    c for c in feat.columns
    if (
        c.startswith("barrel_")
        or c.startswith("mold_")
        or c in [
            "melt_minus_mold_temp",
            "viscosity_proxy_pressure_per_speed",
            "filling_to_injection_ratio",
            "switch_to_injection_pressure_ratio",
            "avg_to_max_back_pressure_ratio",
            "plasticizing_position_per_sec",
        ]
        or "__delta_prev" in c
        or "__prev5_range" in c
    )
]

MODEL_FEATURES = PROCESS_FEATURES + RUN_CONTEXT + DERIVED
MODEL_FEATURES = list(dict.fromkeys([c for c in MODEL_FEATURES if c in feat.columns]))

print("Process features:", len(PROCESS_FEATURES))
print("Run/context:", len(RUN_CONTEXT))
print("Derived:", len(DERIVED))
print("Total model features:", len(MODEL_FEATURES))

pd.DataFrame({
    "feature": MODEL_FEATURES,
    "family": [
        "process" if c in PROCESS_FEATURES
        else ("run_context" if c in RUN_CONTEXT else "derived")
        for c in MODEL_FEATURES
    ]
}).to_csv(OUT_DIR/"V6_02_feature_dictionary.csv", index=False)



## 4. Sign-constrained logistic model

Constraint philosophy:
- larger process deviation / instability should not become "protective" solely due to small-sample noise
- early-run indicator is constrained non-negative
- elapsed time is constrained non-positive
- other raw process settings remain unconstrained

This is intentionally simple and interpretable.


In [ ]:

# ============================================================
# 4. Constrained logistic implementation
# ============================================================
class ConstrainedLogistic:
    def __init__(self, signs=None, l2=1.0, maxiter=1500):
        self.signs = signs or {}
        self.l2 = l2
        self.maxiter = maxiter

    def fit(self, X, y, feature_names):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        self.feature_names_ = list(feature_names)
        self.imputer_ = SimpleImputer(strategy="median")
        self.scaler_ = StandardScaler()

        Xi = self.imputer_.fit_transform(X)
        Xs = self.scaler_.fit_transform(Xi)

        n, p = Xs.shape
        sw = compute_sample_weight(class_weight="balanced", y=y)

        def sigmoid(z):
            z = np.clip(z, -35, 35)
            return 1/(1+np.exp(-z))

        def obj(theta):
            b = theta[0]
            w = theta[1:]
            pr = sigmoid(b + Xs @ w)
            eps = 1e-12
            loss = -np.sum(
                sw * (y*np.log(pr+eps) + (1-y)*np.log(1-pr+eps))
            ) / np.sum(sw)
            return loss + 0.5*self.l2*np.sum(w*w)/max(n,1)

        bounds = [(None,None)]
        for name in self.feature_names_:
            s = int(self.signs.get(name,0))
            if s > 0:
                bounds.append((0,None))
            elif s < 0:
                bounds.append((None,0))
            else:
                bounds.append((None,None))

        init = np.zeros(p+1)
        res = minimize(
            obj, init, method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter":self.maxiter}
        )

        self.intercept_ = float(res.x[0])
        self.coef_ = res.x[1:].copy()
        self.success_ = bool(res.success)
        self.message_ = str(res.message)
        return self

    def predict_proba(self, X):
        Xi = self.imputer_.transform(np.asarray(X,dtype=float))
        Xs = self.scaler_.transform(Xi)
        z = np.clip(self.intercept_ + Xs @ self.coef_, -35, 35)
        p = 1/(1+np.exp(-z))
        return np.column_stack([1-p,p])

# Constraints
SIGN_MAP = {
    "early60": +1,
    "elapsed_min": -1,
    "log_elapsed_min": -1,
}

for c in MODEL_FEATURES:
    if "__prev5_range" in c:
        SIGN_MAP[c] = +1

print("Constrained feature count:", sum(v != 0 for v in SIGN_MAP.values()))



## 5. LOGO + sequential validation and hybrid inspection policy

Policies:
- **Time60**: earlier shots first
- **RiskOnly**: constrained-logistic probability
- **Hybrid60Risk**: every shot within 60 min is ranked ahead of later shots; within each block, use risk
- **UnconstrainedRisk**: standard logistic comparison

We evaluate equal inspection budgets, because that is the operational decision.


In [ ]:

# ============================================================
# 5. OOF predictions: LOGO + sequential
# ============================================================
def fit_unconstrained(Xtr, ytr):
    return Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("clf",LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            C=1.0,
            random_state=SEED
        ))
    ]).fit(Xtr,ytr)

def evaluate_budget_curve(pred_df, validation_name):
    rows = []

    policies = {
        "Time60": "score_time",
        "RiskOnly": "score_risk",
        "Hybrid60Risk": "score_hybrid",
        "UnconstrainedRisk": "score_unconstrained",
    }

    total_fail = int(pred_df["y_true"].sum())

    for allocation in ["global","fixed_per_run"]:
        for policy,score_col in policies.items():
            for budget in BUDGETS:
                if allocation == "global":
                    k = max(1, int(math.ceil(len(pred_df)*budget)))
                    chosen = pred_df.sort_values(
                        [score_col,"timestamp"],
                        ascending=[False,True]
                    ).head(k)
                else:
                    parts = []
                    for run_id,g in pred_df.groupby("run_id"):
                        k = max(1, int(math.ceil(len(g)*budget)))
                        parts.append(
                            g.sort_values(
                                [score_col,"timestamp"],
                                ascending=[False,True]
                            ).head(k)
                        )
                    chosen = pd.concat(parts, ignore_index=True)

                caught = int(chosen["y_true"].sum())
                recall = caught/total_fail if total_fail else np.nan

                rows.append({
                    "validation": validation_name,
                    "allocation": allocation,
                    "policy": policy,
                    "budget": budget,
                    "n_eval": len(pred_df),
                    "n_fail": total_fail,
                    "n_inspected": len(chosen),
                    "actual_inspection_fraction": len(chosen)/len(pred_df),
                    "defects_caught": caught,
                    "defect_recall": recall,
                    "lift_vs_random": recall/budget if budget>0 else np.nan,
                })

    return pd.DataFrame(rows)

def run_logo_predictions(data):
    data = data.reset_index(drop=True).copy()
    X = data[MODEL_FEATURES]
    y = data["_y"].astype(int).to_numpy()
    groups = data["run_id"].astype(str).to_numpy()

    logo = LeaveOneGroupOut()
    out = []

    for fold,(tr,te) in enumerate(logo.split(X,y,groups),1):
        if len(np.unique(y[tr])) < 2:
            continue

        uncon = fit_unconstrained(X.iloc[tr], y[tr])

        cons = ConstrainedLogistic(
            signs=SIGN_MAP, l2=1.0
        ).fit(
            X.iloc[tr].values, y[tr], MODEL_FEATURES
        )

        p_u = uncon.predict_proba(X.iloc[te])[:,1]
        p_c = cons.predict_proba(X.iloc[te].values)[:,1]

        tmp = data.iloc[te][
            ["_timestamp","run_id","_product","_y",
             "elapsed_min","early60","_fail_side"]
        ].copy()

        tmp["fold"] = fold
        tmp["score_unconstrained"] = p_u
        tmp["score_risk"] = p_c
        tmp["score_time"] = -tmp["elapsed_min"].astype(float)
        tmp["score_hybrid"] = 10.0*tmp["early60"] + p_c
        out.append(tmp)

    pred = pd.concat(out, ignore_index=True)
    pred = pred.rename(columns={"_timestamp":"timestamp","_y":"y_true"})
    return pred

def run_sequential_predictions(data, min_prior_runs=3):
    runs = (
        data.groupby("run_id")
        .agg(
            start=("_timestamp","min"),
            product=("_product","first"),
        )
        .reset_index()
        .sort_values("start")
    )

    outs = []

    for i in range(min_prior_runs, len(runs)):
        test_run = runs.iloc[i]["run_id"]
        test_start = runs.iloc[i]["start"]

        train_runs = runs.loc[
            runs["start"] < test_start, "run_id"
        ].tolist()

        tr = data.index[data["run_id"].isin(train_runs)].to_numpy()
        te = data.index[data["run_id"]==test_run].to_numpy()

        if len(te)==0 or len(tr)==0:
            continue

        ytr = data.loc[tr,"_y"].astype(int).to_numpy()
        if len(np.unique(ytr)) < 2:
            continue

        uncon = fit_unconstrained(
            data.loc[tr,MODEL_FEATURES],
            ytr
        )
        cons = ConstrainedLogistic(
            signs=SIGN_MAP,l2=1.0
        ).fit(
            data.loc[tr,MODEL_FEATURES].values,
            ytr,
            MODEL_FEATURES
        )

        p_u = uncon.predict_proba(
            data.loc[te,MODEL_FEATURES]
        )[:,1]
        p_c = cons.predict_proba(
            data.loc[te,MODEL_FEATURES].values
        )[:,1]

        tmp = data.loc[te,[
            "_timestamp","run_id","_product","_y",
            "elapsed_min","early60","_fail_side"
        ]].copy()

        tmp["score_unconstrained"] = p_u
        tmp["score_risk"] = p_c
        tmp["score_time"] = -tmp["elapsed_min"].astype(float)
        tmp["score_hybrid"] = 10.0*tmp["early60"] + p_c
        outs.append(tmp)

    if not outs:
        return pd.DataFrame()

    pred = pd.concat(outs, ignore_index=True)
    pred = pred.rename(columns={"_timestamp":"timestamp","_y":"y_true"})
    return pred

logo_pred = run_logo_predictions(feat)
seq_pred = run_sequential_predictions(feat)

logo_curve = evaluate_budget_curve(logo_pred, "LOGO")
seq_curve = (
    evaluate_budget_curve(seq_pred, "Sequential")
    if len(seq_pred) else pd.DataFrame()
)

curves = pd.concat([logo_curve,seq_curve], ignore_index=True)
curves.to_csv(OUT_DIR/"V6_03_run_aware_budget_curves.csv", index=False)

logo_pred.to_csv(OUT_DIR/"V6_03_logo_predictions.csv", index=False)
seq_pred.to_csv(OUT_DIR/"V6_03_sequential_predictions.csv", index=False)

display(
    curves[
        (curves["allocation"]=="global")
        & (curves["budget"].isin([0.10,0.20]))
    ].sort_values(["validation","budget","defect_recall"],ascending=[True,True,False])
)

# Within-run AUC: diagnostic of whether model separates shots inside a run.
within_rows = []
for run_id,g in logo_pred.groupby("run_id"):
    if g["y_true"].nunique() < 2:
        continue
    for policy,colname in [
        ("RiskOnly","score_risk"),
        ("Hybrid60Risk","score_hybrid"),
        ("Time60","score_time"),
    ]:
        try:
            auc = roc_auc_score(g["y_true"],g[colname])
        except Exception:
            auc = np.nan
        within_rows.append({
            "run_id":run_id,
            "product":g["_product"].iloc[0],
            "policy":policy,
            "within_run_roc_auc":auc,
            "n":len(g),
            "fails":int(g["y_true"].sum()),
        })

within_auc_df = pd.DataFrame(within_rows)
within_auc_df.to_csv(OUT_DIR/"V6_03_within_run_auc.csv", index=False)
display(
    within_auc_df.groupby("policy")["within_run_roc_auc"]
    .agg(["mean","median","count"])
)



## 6. Side model: static RH rule vs learned cavity priority

The report used RH-first as a strong operational rule.
v6 tests whether a learned side model adds anything beyond that rule.

Evaluation is restricted to **one-sided defective shots**, because those are the shots
where "which side first?" has an observable ground truth.


In [ ]:

# ============================================================
# 6. LOGO side prediction
# ============================================================
side_eval = feat[
    (feat["_y"]==1)
    & (feat["_fail_side"].isin(["LH","RH"]))
].copy().reset_index(drop=True)

SIDE_FEATURES = [
    c for c in PROCESS_FEATURES + RUN_CONTEXT + DERIVED
    if c in side_eval.columns
]

side_eval["side_y"] = (side_eval["_fail_side"]=="RH").astype(int)

side_oof_rows = []

if RUN_SIDE_MODEL and len(side_eval) >= 10:
    logo = LeaveOneGroupOut()

    X = side_eval[SIDE_FEATURES]
    y = side_eval["side_y"].to_numpy()
    groups = side_eval["run_id"].astype(str).to_numpy()

    for fold,(tr,te) in enumerate(logo.split(X,y,groups),1):
        ytr = y[tr]

        if len(np.unique(ytr)) < 2:
            # fallback: training majority side
            p_rh = np.repeat(ytr.mean() if len(ytr) else 0.5, len(te))
        else:
            m = Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("sc",StandardScaler()),
                ("clf",LogisticRegression(
                    class_weight="balanced",
                    C=0.5,
                    max_iter=5000,
                    random_state=SEED
                ))
            ])
            m.fit(X.iloc[tr],ytr)
            p_rh = m.predict_proba(X.iloc[te])[:,1]

        for idx,p in zip(te,p_rh):
            side_oof_rows.append({
                "row_index":int(idx),
                "run_id":side_eval.loc[idx,"run_id"],
                "product":side_eval.loc[idx,"_product"],
                "true_side":side_eval.loc[idx,"_fail_side"],
                "side_y":int(side_eval.loc[idx,"side_y"]),
                "p_RH":float(p),
                "pred_side":"RH" if p>=.5 else "LH",
            })

side_oof = pd.DataFrame(side_oof_rows)

if len(side_oof):
    static_rh_acc = (side_oof["true_side"]=="RH").mean()
    learned_acc = (side_oof["pred_side"]==side_oof["true_side"]).mean()

    side_summary = pd.DataFrame([
        {
            "method":"Random_50_50",
            "side_hit_at_1":0.50,
            "n":len(side_oof),
        },
        {
            "method":"Static_RH_First",
            "side_hit_at_1":static_rh_acc,
            "n":len(side_oof),
        },
        {
            "method":"Learned_LOGO_Logistic",
            "side_hit_at_1":learned_acc,
            "n":len(side_oof),
        },
    ])

    side_oof.to_csv(OUT_DIR/"V6_04_side_oof_predictions.csv",index=False)
    side_summary.to_csv(OUT_DIR/"V6_04_side_priority_summary.csv",index=False)

    display(side_summary)
    display(
        side_oof.groupby(["product","true_side"])
        .size().to_frame("n")
    )
else:
    print("Side-model evaluation unavailable.")



## 7. Split-conformal side prediction

A point classifier always chooses LH or RH.
The conformal layer can instead output:

- `{RH}` → inspect RH only
- `{LH}` → inspect LH only
- `{LH, RH}` → model is uncertain, inspect both

This turns uncertainty into an inspection action.


In [ ]:

# ============================================================
# 7. Group-aware split conformal, evaluated by held-out run
# ============================================================
def conformal_quantile(scores, alpha):
    scores = np.sort(np.asarray(scores,dtype=float))
    n = len(scores)
    if n == 0:
        return 1.0
    rank = int(np.ceil((n+1)*(1-alpha)))
    rank = min(max(rank,1),n)
    return float(scores[rank-1])

def fit_side_base(train_df):
    X = train_df[SIDE_FEATURES]
    y = train_df["side_y"].astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        return None, float(y.mean()) if len(y) else 0.5

    model = Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("clf",LogisticRegression(
            class_weight="balanced",
            C=0.5,
            max_iter=5000,
            random_state=SEED
        ))
    ])
    model.fit(X,y)
    return model,None

def predict_p_rh(model,fallback,dfx):
    if model is None:
        return np.repeat(fallback,len(dfx))
    return model.predict_proba(dfx[SIDE_FEATURES])[:,1]

conformal_rows = []

if RUN_CONFORMAL:
    one_side = side_eval.copy()
    unique_runs = sorted(one_side["run_id"].unique())

    for test_run in unique_runs:
        train_pool = one_side[one_side["run_id"]!=test_run].copy()
        test_def = one_side[one_side["run_id"]==test_run].copy()

        if len(test_def)==0 or train_pool["run_id"].nunique()<3:
            continue

        run_ids = np.array(sorted(train_pool["run_id"].unique()))

        # Find a valid deterministic group split.
        split_found = False
        for attempt in range(20):
            gss = GroupShuffleSplit(
                n_splits=1,
                test_size=0.30,
                random_state=SEED + attempt
            )
            tr_idx,cal_idx = next(
                gss.split(
                    train_pool,
                    train_pool["side_y"],
                    groups=train_pool["run_id"]
                )
            )
            fit_df = train_pool.iloc[tr_idx].copy()
            cal_df = train_pool.iloc[cal_idx].copy()

            if (
                len(cal_df)>=3
                and len(fit_df)>=5
                and fit_df["side_y"].nunique()>=2
            ):
                split_found=True
                break

        if not split_found:
            continue

        model,fallback = fit_side_base(fit_df)

        p_cal = predict_p_rh(model,fallback,cal_df)
        p_true_cal = np.where(
            cal_df["side_y"].to_numpy()==1,
            p_cal,
            1-p_cal
        )
        nonconf = 1-p_true_cal

        for alpha in [0.10,0.05]:
            q = conformal_quantile(nonconf,alpha)

            p_test = predict_p_rh(model,fallback,test_def)
            for (_,row),p in zip(test_def.iterrows(),p_test):
                p_lh = 1-p
                include_lh = (1-p_lh) <= q
                include_rh = (1-p) <= q

                # Conservative fallback: never return empty set.
                if not include_lh and not include_rh:
                    include_lh = True
                    include_rh = True

                pred_set = (
                    "{LH,RH}" if include_lh and include_rh
                    else ("{RH}" if include_rh else "{LH}")
                )

                true_side = row["_fail_side"]
                covered = (
                    (true_side=="LH" and include_lh)
                    or (true_side=="RH" and include_rh)
                )

                conformal_rows.append({
                    "test_run":test_run,
                    "product":row["_product"],
                    "alpha":alpha,
                    "true_side":true_side,
                    "p_RH":float(p),
                    "q":q,
                    "include_LH":bool(include_lh),
                    "include_RH":bool(include_rh),
                    "prediction_set":pred_set,
                    "set_size":int(include_lh)+int(include_rh),
                    "covered":bool(covered),
                })

conformal_df = pd.DataFrame(conformal_rows)

if len(conformal_df):
    conformal_summary = (
        conformal_df.groupby("alpha")
        .agg(
            n=("covered","size"),
            empirical_coverage=("covered","mean"),
            singleton_rate=("set_size",lambda x:(x==1).mean()),
            both_rate=("set_size",lambda x:(x==2).mean()),
            avg_set_size=("set_size","mean"),
        )
        .reset_index()
    )

    conformal_df.to_csv(
        OUT_DIR/"V6_05_conformal_side_predictions.csv",
        index=False
    )
    conformal_summary.to_csv(
        OUT_DIR/"V6_05_conformal_side_summary.csv",
        index=False
    )
    display(conformal_summary)
else:
    print("Conformal side evaluation unavailable.")



## 8. Hierarchical Identifiability-Aware Inspection (IAHI)

**Stage 1 — shot risk**
- use the run-aware hybrid score

**Stage 2 — side priority**
- learned side model

**Stage 3 — uncertainty action**
- conformal singleton → inspect one side
- conformal doubleton → inspect both sides

We compare the number of part inspections required at the same shot-selection budget.


In [ ]:

# ============================================================
# 8. Hierarchical policy
# ============================================================
# Build LOGO conformal side predictions for ALL shots so that inspection cost
# can be evaluated on passes as well. The conformal threshold is calibrated
# only on one-sided defective training shots.
all_side_policy_rows = []

if RUN_HIERARCHICAL_POLICY and len(side_eval)>=10:
    all_runs = sorted(feat["run_id"].unique())

    for test_run in all_runs:
        defect_train_pool = side_eval[
            side_eval["run_id"]!=test_run
        ].copy()

        test_all = feat[
            feat["run_id"]==test_run
        ].copy()

        if (
            len(test_all)==0
            or defect_train_pool["run_id"].nunique()<3
        ):
            # conservative fallback: both
            for idx,row in test_all.iterrows():
                all_side_policy_rows.append({
                    "shot_index":idx,
                    "run_id":test_run,
                    "p_RH":0.5,
                    "include_LH":True,
                    "include_RH":True,
                    "set_size":2,
                    "alpha":0.10,
                })
            continue

        split_found=False
        for attempt in range(20):
            gss = GroupShuffleSplit(
                n_splits=1,test_size=.30,
                random_state=SEED+100+attempt
            )
            tr_idx,cal_idx = next(
                gss.split(
                    defect_train_pool,
                    defect_train_pool["side_y"],
                    groups=defect_train_pool["run_id"]
                )
            )
            fit_df = defect_train_pool.iloc[tr_idx].copy()
            cal_df = defect_train_pool.iloc[cal_idx].copy()
            if (
                len(fit_df)>=5
                and fit_df["side_y"].nunique()>=2
                and len(cal_df)>=3
            ):
                split_found=True
                break

        if not split_found:
            for idx,row in test_all.iterrows():
                all_side_policy_rows.append({
                    "shot_index":idx,
                    "run_id":test_run,
                    "p_RH":0.5,
                    "include_LH":True,
                    "include_RH":True,
                    "set_size":2,
                    "alpha":0.10,
                })
            continue

        model,fallback = fit_side_base(fit_df)

        p_cal = predict_p_rh(model,fallback,cal_df)
        p_true_cal = np.where(
            cal_df["side_y"].to_numpy()==1,
            p_cal,
            1-p_cal
        )
        q = conformal_quantile(1-p_true_cal, alpha=.10)

        p_test = predict_p_rh(model,fallback,test_all)

        for (idx,row),p in zip(test_all.iterrows(),p_test):
            p_lh = 1-p
            inc_lh = (1-p_lh) <= q
            inc_rh = (1-p) <= q

            if not inc_lh and not inc_rh:
                inc_lh=inc_rh=True

            all_side_policy_rows.append({
                "shot_index":idx,
                "run_id":test_run,
                "p_RH":float(p),
                "include_LH":bool(inc_lh),
                "include_RH":bool(inc_rh),
                "set_size":int(inc_lh)+int(inc_rh),
                "alpha":0.10,
            })

side_policy_all = pd.DataFrame(all_side_policy_rows)

# Map LOGO shot risk prediction back to feat by run/timestamp.
risk_map = logo_pred.copy()
risk_map["_merge_key"] = (
    risk_map["run_id"].astype(str)
    + "||"
    + risk_map["timestamp"].astype(str)
)

eval_all = feat.copy()
eval_all["_merge_key"] = (
    eval_all["run_id"].astype(str)
    + "||"
    + eval_all["_timestamp"].astype(str)
)

eval_all = eval_all.merge(
    risk_map[[
        "_merge_key","score_hybrid","score_risk"
    ]],
    on="_merge_key",
    how="left"
)

eval_all = eval_all.reset_index().rename(columns={"index":"shot_index"})
eval_all = eval_all.merge(
    side_policy_all,
    on=["shot_index","run_id"],
    how="left"
)

eval_all["include_LH"] = eval_all["include_LH"].fillna(True)
eval_all["include_RH"] = eval_all["include_RH"].fillna(True)
eval_all["set_size"] = eval_all["set_size"].fillna(2)

def side_captures(row, policy):
    fs = row["_fail_side"]
    if row["_y"]==0:
        return 0

    if policy=="BothSides":
        return 1
    if policy=="RHOnly":
        return int(fs in ["RH","BOTH"])
    if policy=="PointSide":
        pred = "RH" if row.get("p_RH",.5)>=.5 else "LH"
        return int(fs=="BOTH" or fs==pred)
    if policy=="Conformal":
        if fs=="BOTH":
            return int(row["include_LH"] or row["include_RH"])
        if fs=="LH":
            return int(row["include_LH"])
        if fs=="RH":
            return int(row["include_RH"])
    return 0

hier_rows = []

for budget in BUDGETS:
    eligible = eval_all[eval_all["score_hybrid"].notna()].copy()
    k = max(1,int(math.ceil(len(eligible)*budget)))
    selected = eligible.sort_values(
        ["score_hybrid","_timestamp"],
        ascending=[False,True]
    ).head(k).copy()

    for policy in ["BothSides","RHOnly","PointSide","Conformal"]:
        tmp = selected.copy()
        tmp["captured"] = tmp.apply(
            lambda r: side_captures(r,policy), axis=1
        )

        if policy=="BothSides":
            part_inspections = 2*len(tmp)
        elif policy in ["RHOnly","PointSide"]:
            part_inspections = len(tmp)
        else:
            part_inspections = int(tmp["set_size"].sum())

        defects_captured = int(
            tmp.loc[tmp["_y"]==1,"captured"].sum()
        )
        total_defects = int(eligible["_y"].sum())

        hier_rows.append({
            "shot_budget":budget,
            "side_policy":policy,
            "selected_shots":len(tmp),
            "part_inspections":part_inspections,
            "part_inspection_ratio_vs_both":
                part_inspections/(2*len(tmp)),
            "defects_captured":defects_captured,
            "total_defects":total_defects,
            "overall_defect_recall":
                defects_captured/total_defects if total_defects else np.nan,
            "defects_per_100_part_inspections":
                100*defects_captured/max(part_inspections,1),
        })

hier_df = pd.DataFrame(hier_rows)
hier_df.to_csv(
    OUT_DIR/"V6_06_hierarchical_inspection_policy.csv",
    index=False
)
display(hier_df)



## 9. Value of Information (VOI)

We separate two questions:

1. **Does a context variable remove ambiguity within an identical process condition?**
2. **Does it improve out-of-group prediction or inspection efficiency?**

This avoids ranking a variable highly merely because it uniquely identifies rows.


In [ ]:

# ============================================================
# 9. VOI: side, run context, process risk
# ============================================================
def binary_entropy(p):
    p = np.asarray(p,dtype=float)
    out = np.zeros_like(p)
    m = (p>0)&(p<1)
    out[m] = -p[m]*np.log2(p[m])-(1-p[m])*np.log2(1-p[m])
    return out

def exact_entropy(data, process_cols, context_cols=None):
    context_cols = context_cols or []
    cols = process_cols + context_cols
    tmp = data.dropna(subset=cols+["_y"]).copy()
    g = (
        tmp.groupby(cols)["_y"]
        .agg(["size","sum"])
        .reset_index()
    )
    g["p"] = g["sum"]/g["size"]
    g["h"] = binary_entropy(g["p"].values)
    return float(np.average(g["h"],weights=g["size"])) if len(g) else np.nan

voi_rows = []

for product in ["CN7","RG3"]:
    d = df[df["_product"]==product].copy()

    h_x = exact_entropy(d, PROCESS_FEATURES, [])

    # SIDE is part-level context and is the main identifiability candidate.
    h_side = exact_entropy(d, PROCESS_FEATURES, ["_side"])

    voi_rows.append({
        "product":product,
        "context":"SIDE",
        "H_Y_given_X":h_x,
        "H_Y_given_XC":h_side,
        "entropy_reduction":h_x-h_side,
        "relative_entropy_reduction":
            (h_x-h_side)/h_x if h_x>0 else np.nan,
        "interpretation":"part/cavity identity",
    })

# Operational gain of adding run context through the hybrid policy
for validation in curves["validation"].unique():
    sub = curves[
        (curves["validation"]==validation)
        & (curves["allocation"]=="global")
        & (np.isclose(curves["budget"],.20))
    ]

    risk = sub[sub["policy"]=="RiskOnly"]
    hybrid = sub[sub["policy"]=="Hybrid60Risk"]

    if len(risk) and len(hybrid):
        voi_rows.append({
            "product":"ALL",
            "context":f"RUN_CONTEXT_60MIN__{validation}",
            "H_Y_given_X":np.nan,
            "H_Y_given_XC":np.nan,
            "entropy_reduction":np.nan,
            "relative_entropy_reduction":np.nan,
            "interpretation":
                "operational Recall@20 gain = "
                f"{float(hybrid.iloc[0]['defect_recall']-risk.iloc[0]['defect_recall']):.4f}",
        })

voi_df = pd.DataFrame(voi_rows)
voi_df.to_csv(OUT_DIR/"V6_07_value_of_information.csv",index=False)
display(voi_df)



## 10. Optional external cavity-pressure evidence — RWTH Aachen

The 2026 RWTH dataset contains screw-antechamber pressure, **cavity pressure**,
controller output, screw velocity, screw-volume trajectories, and part-mass control records.

This cell deliberately starts with a schema profiler.
If a table with both part mass and cavity-pressure features is automatically discoverable,
the notebook exports candidate files/columns for the next ablation.

Set `AUTO_DOWNLOAD_RWTH=True` after the core v6 run succeeds.


In [ ]:

# ============================================================
# 10. Optional RWTH automatic download + schema profile
# ============================================================
rwth_profile_rows = []

if AUTO_DOWNLOAD_RWTH:
    rwth_zip = EXT_DIR/"RWTH_ExperimentalData.zip"
    rwth_dir = EXT_DIR/"rwth_data"

    try:
        urllib.request.urlretrieve(RWTH_URL, rwth_zip)

        if rwth_dir.exists():
            shutil.rmtree(rwth_dir)
        rwth_dir.mkdir(parents=True,exist_ok=True)

        with zipfile.ZipFile(rwth_zip,"r") as z:
            z.extractall(rwth_dir)

        for p in rwth_dir.rglob("*"):
            if not p.is_file():
                continue

            row = {
                "file":str(p.relative_to(rwth_dir)),
                "size_mb":p.stat().st_size/1024/1024,
                "suffix":p.suffix.lower(),
            }

            if p.suffix.lower() in [".csv",".parquet",".pq",".xlsx",".xls"]:
                try:
                    if p.suffix.lower()==".csv":
                        t = pd.read_csv(p,nrows=20)
                    elif p.suffix.lower() in [".parquet",".pq"]:
                        t = pd.read_parquet(p)
                        if len(t)>20:
                            t=t.head(20)
                    else:
                        t = pd.read_excel(p,nrows=20)

                    cols = list(map(str,t.columns))
                    row["n_columns"] = len(cols)
                    row["mass_columns"] = "|".join([
                        c for c in cols
                        if any(k in canonical_name(c)
                               for k in ["mass","weight","gewicht"])
                    ])
                    row["cavity_pressure_columns"] = "|".join([
                        c for c in cols
                        if "cavity" in canonical_name(c)
                        and "press" in canonical_name(c)
                    ])
                    row["pressure_columns"] = "|".join([
                        c for c in cols
                        if "press" in canonical_name(c)
                    ][:30])
                except Exception as e:
                    row["read_error"]=repr(e)

            rwth_profile_rows.append(row)

        rwth_profile_df = pd.DataFrame(rwth_profile_rows)
        rwth_profile_df.to_csv(
            OUT_DIR/"V6_08_rwth_schema_profile.csv",
            index=False
        )
        display(
            rwth_profile_df.sort_values("size_mb",ascending=False).head(50)
        )

    except Exception as e:
        print("RWTH download/profile failed:",repr(e))
else:
    print("AUTO_DOWNLOAD_RWTH=False")



## 11. Final v6 novelty summary and ZIP

The summary intentionally distinguishes:
- run-level operational gain,
- side-priority gain,
- conformal coverage / inspection reduction,
- information gain from side context.


In [ ]:

# ============================================================
# 11. Summary / package
# ============================================================
summary_rows = []

for validation in curves["validation"].unique():
    for budget in [0.10,0.20]:
        sub = curves[
            (curves["validation"]==validation)
            & (curves["allocation"]=="global")
            & np.isclose(curves["budget"],budget)
        ]
        for policy in ["Time60","RiskOnly","Hybrid60Risk"]:
            r = sub[sub["policy"]==policy]
            if len(r):
                summary_rows.append({
                    "section":"run_aware",
                    "metric":f"{validation}_{policy}_Recall@{int(100*budget)}",
                    "value":float(r.iloc[0]["defect_recall"]),
                })

if "side_summary" in globals():
    for _,r in side_summary.iterrows():
        summary_rows.append({
            "section":"side_priority",
            "metric":r["method"],
            "value":float(r["side_hit_at_1"]),
        })

if "conformal_summary" in globals():
    for _,r in conformal_summary.iterrows():
        summary_rows.extend([
            {
                "section":"conformal",
                "metric":f"coverage_alpha_{r['alpha']}",
                "value":float(r["empirical_coverage"]),
            },
            {
                "section":"conformal",
                "metric":f"singleton_rate_alpha_{r['alpha']}",
                "value":float(r["singleton_rate"]),
            },
        ])

novelty_summary = pd.DataFrame(summary_rows)
novelty_summary.to_csv(
    OUT_DIR/"FINAL_v6_novelty_summary.csv",
    index=False
)
display(novelty_summary)

# Source note
source_note = {
    "legacy_kamp_repo":LEGACY_KAMP_REPO,
    "rwth_dataset":RWTH_URL,
    "seed":SEED,
    "framework":"IAHI - Identifiability-Aware Hierarchical Inspection",
}
(OUT_DIR/"run_config_v6_novelty.json").write_text(
    json.dumps(source_note,indent=2,ensure_ascii=False)
)

# Manifest
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

manifest=[]
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "file":str(p.relative_to(OUT_DIR)),
            "size_bytes":p.stat().st_size,
            "sha256":sha256_file(p),
        })

pd.DataFrame(manifest).to_csv(
    OUT_DIR/"result_manifest_v6.csv",
    index=False
)

zip_path = shutil.make_archive(
    "/content/KAMP01_v6_novelty_results",
    "zip",
    OUT_DIR
)
print("Result ZIP:",zip_path)

from google.colab import files
files.download(zip_path)
